# 01 — Il classificatore a feature con le etichette MBFC (tentativo 10)

Il TF-IDF + regressione logistica dei tentativi 06–09, invariato, con l'etichetta **factual reporting MBFC** (Very
Low, Low, Mixed, Mostly Factual, High) al posto del livello NewsGuard, su una delle due versioni del dataset
(`VERSIONE` = `grezzo` o `pulito`, da `2-dataset/01_dataset_mbfc`; stesse testate valutate e stessi fold).

- Nel training anche le testate di solo training, ognuna con il suo **peso** (un gruppo vale al più 5 testate); le
  classi si bilanciano sui pesi. Test e metriche solo sulle **222 testate valutate**.
- Fold **per gruppo** (reti ∪ owner ∪ Parent Domain). Vocabolario e IDF solo sul training del fold.
- **C** scelto tra 0,1, 1, 10 e 100 con una validazione interna a 3 fold per gruppi sul solo training (F1 macro per
  testata), come il riferimento del tentativo 09.
- Per testata: **voto di maggioranza** dei suoi articoli. Binario: affidabile = almeno *Mostly Factual*.
- Due baseline senza testo: la classe di maggioranza e una scelta a caso con le proporzioni del training.

Si salva tutto (`<NOME>_<VERSIONE>_fold_gruppo_articoli.csv` con logit e probabilità per articolo, `..._metriche.json`,
`<NOME>_<VERSIONE>_metriche.csv`, `..._FATTO`) in `Results/3-classificazione/`.


In [1]:
import json
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold

from comune import CLASSI, NOME_SCHEMA, SCHEMI, SEED, carica, cartella_risultati, letture, metriche, per_testata, pesi_bilanciati, registratore, LIVELLI

In [2]:
NOME = "feature_tuned"
VERSIONE = "pulito"         # grezzo / pulito
TAGLIO_TOKEN = 0           # 0 = testo intero; N = primi N token (tokenizer di ModernBERT)
C_GRIGLIA = "0.1,1,10,100" # "" = C fisso a 1; "0.1,1,10,100" = C scelto con una validazione interna per gruppi (solo training)
SCHEMI_DA_FARE = SCHEMI
SOLO_VALUTAZIONE = False   # True: training solo sulle testate valutate (come la prima versione)
N_JOBS = 5                 # fold in parallelo
PROVA = False              # True: poche testate, risultati in Results/3-classificazione/prova/

In [3]:
# Parameters
VERSIONE = "grezzo"
N_JOBS = 5


In [4]:
NOME = f"{NOME}_{VERSIONE}"
OUT = cartella_risultati(PROVA)
registra = registratore(OUT / f"avanzamento_{NOME}.log")
art = carica(VERSIONE, PROVA, SOLO_VALUTAZIONE)
if TAGLIO_TOKEN:
    from transformers import AutoTokenizer
    tok = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")
    enc = tok(art.text.tolist(), truncation=True, max_length=TAGLIO_TOKEN, add_special_tokens=False, return_offsets_mapping=True)
    art["text"] = [t[:o[-1][1]] if o else t for t, o in zip(art.text, enc["offset_mapping"])]
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate ({art[art.valutazione].domain.nunique()} valutate), "
         f"taglio {TAGLIO_TOKEN or 'nessuno'}, solo valutate {SOLO_VALUTAZIONE} ===")

2026-10-23 12:51:33  === feature_tuned_grezzo: 61219 articoli, 456 testate (222 valutate), taglio nessuno, solo valutate False ===


## Addestramento

Per ogni fold: `fit_transform` sul training, `transform` sul test. I logit sono la `decision_function` della
regressione logistica.

In [5]:
def nuovo_tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def scegli_C(tr):
    """C con la F1 macro per testata (voto) più alta in una validazione a 3 fold per gruppi, sul solo training."""
    griglia = [float(x) for x in C_GRIGLIA.split(",")]
    testate = tr.drop_duplicates("domain").reset_index(drop=True)
    punteggi = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(testate, testate.label, testate.gruppo):
        a = tr[tr.domain.isin(testate.domain[itr])]
        b = tr[tr.domain.isin(testate.domain[iva]) & tr.valutazione]          # si valuta come nel test: solo testate valutate
        v = nuovo_tfidf(); Xa, Xb = v.fit_transform(a.text), v.transform(b.text)
        pa = pesi_bilanciati(a.label, a.peso)
        for c in griglia:
            p = LogisticRegression(C=c, max_iter=1000, random_state=SEED).fit(Xa, a.label, sample_weight=pa).predict(Xb)
            t = per_testata(pd.DataFrame({"domain": b.domain.values, "y_true": b.label.values, "y_pred": p}))
            punteggi[c].append(f1_score(t.y_true, t.y_pred, average="macro", labels=range(5), zero_division=0))
    medie = {c: float(np.mean(s)) for c, s in punteggi.items()}
    return max(medie, key=medie.get), medie


def un_fold(schema, fold):
    t0 = time.time()
    tr, te = art[art[schema] != fold], art[(art[schema] == fold) & art.valutazione]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    if C_GRIGLIA:
        registra(f"{schema} fold {fold}: C scelto {C} (F1 interne {', '.join(f'{k}: {v:.3f}' for k, v in medie.items())})")
    v = nuovo_tfidf()
    clf = LogisticRegression(C=C, max_iter=1000, random_state=SEED)
    clf.fit(v.fit_transform(tr.text), tr.label, sample_weight=pesi_bilanciati(tr.label, tr.peso))   # vocabolario, IDF e modello: solo training
    X = v.transform(te.text)                                   # test: solo transform
    registra(f"{schema} fold {fold}: fatto in {time.time() - t0:.0f} s ({len(tr)} training, {len(te)} test)")
    return te.index.to_numpy(), clf.classes_, clf.decision_function(X), clf.predict_proba(X)


def addestra(schema):
    file_out = OUT / f"{NOME}_{schema}_articoli.csv"
    if file_out.exists():
        registra(f"{schema}: già fatto, carico {file_out.name}")
        return pd.read_csv(file_out)
    logit, proba = np.full((len(art), 5), np.nan), np.zeros((len(art), 5))
    for idx, classi, z, p in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(schema, f) for f in sorted(art[schema].unique())):
        logit[idx[:, None], classi[None, :]] = z
        proba[idx[:, None], classi[None, :]] = p
    pred = pd.DataFrame({"article_id": art.article_id, "domain": art.domain, "fold": art[schema], "y_true": art.label})
    for k in range(5):
        pred[f"logit_l{k + 1}"] = logit[:, k]
    for k, c in enumerate(CLASSI):
        pred[c] = proba[:, k]
    pred["y_pred"] = proba.argmax(1)
    pred = pred[art.valutazione.to_numpy()].reset_index(drop=True)      # solo le testate valutate
    pred.to_csv(file_out, index=False)
    json.dump(metriche(pred), open(OUT / f"{NOME}_{schema}_metriche.json", "w"), indent=1)
    return pred


pred = {s: addestra(s) for s in SCHEMI_DA_FARE}

2026-10-23 13:02:47  fold_gruppo fold 2: C scelto 0.1 (F1 interne 0.1: 0.283, 1.0: 0.265, 10.0: 0.265, 100.0: 0.264)


2026-10-23 13:03:00  fold_gruppo fold 5: C scelto 1.0 (F1 interne 0.1: 0.287, 1.0: 0.288, 10.0: 0.286, 100.0: 0.280)


2026-10-23 13:03:22  fold_gruppo fold 3: C scelto 0.1 (F1 interne 0.1: 0.275, 1.0: 0.252, 10.0: 0.240, 100.0: 0.224)


2026-10-23 13:03:24  fold_gruppo fold 4: C scelto 0.1 (F1 interne 0.1: 0.317, 1.0: 0.251, 10.0: 0.248, 100.0: 0.257)


2026-10-23 13:03:49  fold_gruppo fold 1: C scelto 1.0 (F1 interne 0.1: 0.256, 1.0: 0.264, 10.0: 0.259, 100.0: 0.250)


2026-10-23 13:05:01  fold_gruppo fold 2: fatto in 804 s (46810 training, 7216 test)


## Baseline senza testo

Per ogni fold, dalle testate valutate di training (come nella prima versione): la classe più frequente, e una classe estratta a caso con le
proporzioni del training (una per testata, seed fisso).

In [ ]:
def baseline(schema, modo):
    rng = np.random.default_rng(SEED)
    y = np.full(len(art), -1)
    for fold in sorted(art[schema].unique()):
        tr, te = art[(art[schema] != fold) & art.valutazione], art[(art[schema] == fold) & art.valutazione]
        priori = tr.drop_duplicates("domain").label.value_counts(normalize=True).sort_index()
        if modo == "maggioranza":
            y[te.index] = priori.idxmax()
        else:
            scelte = dict(zip(te.domain.unique(), rng.choice(priori.index, te.domain.nunique(), p=priori.values)))
            y[te.index] = te.domain.map(scelte)
    b = pd.DataFrame({"domain": art.domain, "y_true": art.label, "y_pred": y})
    return b[art.valutazione.to_numpy()].reset_index(drop=True)


for s in SCHEMI_DA_FARE:
    for modo in ["maggioranza", "a caso"]:
        json.dump(metriche(baseline(s, modo)), open(OUT / f"baseline_{modo.replace(' ', '_')}_{s}_metriche.json", "w"), indent=1)

## Risultati

Per articolo e per testata (voto di maggioranza): accuratezza a 5 classi, F1 macro, binario e tolerant.

In [ ]:
righe = []
for s in SCHEMI_DA_FARE:
    righe.append({"modello": f"TF-IDF + LR ({NOME})", "fold": NOME_SCHEMA[s], **letture(pred[s])})
    for modo in ["maggioranza", "a caso"]:
        righe.append({"modello": f"baseline: {modo}", "fold": NOME_SCHEMA[s], **letture(baseline(s, modo))})
ris = pd.DataFrame(righe).set_index(["fold", "modello"])
ris.to_csv(OUT / f"{NOME}_metriche.csv")
ris[[c for c in ris.columns if c.startswith("testata")] + ["articolo 5cl", "articolo bin"]]

In [ ]:
t = per_testata(pred[SCHEMI_DA_FARE[0]])
LIV = LIVELLI
display(pd.crosstab(t.y_true.map(dict(enumerate(LIV))), t.y_pred.map(dict(enumerate(LIV))), rownames=["vero"], colnames=["predetto"]))
(OUT / f"{NOME}_FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))